In [5]:
import pandas as pd

df=pd.read_csv('messy_employee_data.csv')
#df.info()

#print(df['email'].unique())
#print(df['join_date'].unique())

print(df['emp_id'].duplicated().sum())

df['age'] = df['age'].str.replace(' yrs', '', regex=False)
df['age'] = df['age'].replace(['NA', 'N/A', 'unknown', ''], pd.NA)   # catch ALL missing-value variants
df['age'] = pd.to_numeric(df['age'], errors='coerce')                  # coerce = don't crash on leftovers
df.loc[(df['age'] < 18) | (df['age'] > 65), 'age'] = pd.NA               # outliers -> NaN, not dropped
print(df['age'].isna().sum())                                              # should now show a real count > 0
df['age'] = df['age'].fillna(df['age'].median())                            # NOW fillna actually does something

df['salary']=df['salary'].str.replace('₹','')
df['salary']=df['salary'].replace(['NA','N/A','unknown',''],pd.NA)
df['salary']=pd.to_numeric(df['salary'],errors='coerce')

df.loc[df['salary']<0,'salary']=pd.NA
df['salary']=df['salary'].fillna(df['salary'].median())
#print(df['salary'].unique())

df['city']=df['city'].str.title().str.strip()
df['city']=df['city'].replace('Bengaluru','Bangalore')
df['department']=df['department'].str.title().str.strip()
df['department'] = df['department'].replace('It', 'IT')


df['join_date'] = pd.to_datetime(df['join_date'], errors='coerce', format='%d-%m-%Y')


df['email'] = df['email'].str.lower()
df['email_valid'] = df['email'].str.match(r'^[\w\.]+@[\w\.]+\.\w+$', na=False)
#print(df['email_valid'].value_counts())
#print(df[~df['email_valid']]['email'].unique())

print("Exact duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("Shape after removing exact duplicates:", df.shape)

print("Duplicate full_name + same city/department combo:")
print(df[df.duplicated(subset=['full_name', 'city', 'department'], keep=False)].sort_values('full_name'))

df.reset_index(drop=True)
df.to_csv("cleaned_employee.csv")

13
290
Exact duplicate rows: 8
Shape after removing exact duplicates: (405, 9)
Duplicate full_name + same city/department combo:
     emp_id     full_name   age    salary       city join_date  \
101    1165    Amit Kumar  38.0  131789.0       Pune       NaT   
230    1033    Amit Kumar  38.0   51885.0  Bangalore       NaT   
162    1299    Amit Kumar  38.0   84990.0  Bangalore       NaT   
314    1075    Amit Kumar  18.0   84990.0     Mumbai       NaT   
39     1006    Amit Kumar  38.0  109049.0     Mumbai       NaT   
..      ...           ...   ...       ...        ...       ...   
297    1140  Vikram Singh  38.0  136756.0     Mumbai       NaT   
368    1380  Vikram Singh  49.0   66835.0     Mumbai       NaT   
195    1135  Vikram Singh  38.0   60473.0  Bangalore       NaT   
199    1142  Vikram Singh  38.0   84990.0       Pune       NaT   
327    1183  Vikram Singh  38.0   84990.0  Bangalore       NaT   

                      email department  email_valid  
101           invalid_em